# Model Context Protocol (MCP)

**WatSPEED Agentic AI prep — Week 5 - MCP**

Runs offline. Set `OPENAI_API_KEY` to swap the stub model for a real one.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() if (pathlib.Path.cwd() / 'agentkit.py').exists()
                      else pathlib.Path.cwd() / 'notebooks'))
from agentkit import *

## Model Context Protocol

MCP is on the syllabus, and it answers a question the earlier notebooks left open:
your tools were Python functions *in the same process as the agent*. MCP makes tools a
**service** the agent connects to — so one tool server works with any MCP-speaking client
(Claude Desktop, Claude Code, an agent you write), in any language.

The protocol is JSON-RPC 2.0. Three methods carry most of the weight:

| Method | Meaning |
|---|---|
| `tools/list` | what can I call? |
| `tools/call` | run one |
| `resources/read` | read a document/dataset by URI |

### The Restaurant Analogy: What MCP Actually Is

> **A Common Misconception:** Many analysts assume MCP translates Python code into TypeScript so an LLM can run it. This is false. MCP is just a live communication protocol (like a telephone line).
>
> **How it works:**
> 1. **The Kitchen (Your Python Server):** You write your complex SAS-equivalent statistical functions in Python and turn on the server.
> 2. **The Menu (JSON Schema):** The Python server publishes a 'Menu' describing its tools.
> 3. **The Waiter (The LLM Agent):** An agent (written in TypeScript, Python, or Go) reads the Menu. It doesn't need to read or understand your Python source code. It treats your code like a black box.
> 4. **The Order:** The agent asks the server to execute a tool and return the result.
>
> **The RAP Value:** You can write pure, heavy data science code in Python, and *any* software engineer can use your tools from their TypeScript web app simply by talking to your MCP server!


In [2]:
import json

# A server's reply to tools/list. Note it is the same JSON Schema as notebook 02 -
# MCP standardises the transport, not the schema.
tools_list_response = {
    "jsonrpc": "2.0", "id": 1,
    "result": {"tools": [
        {"name": "crosstab",
         "description": "Cross-tabulate two survey variables",
         "inputSchema": schema(row_var="string", col_var="string")},
        {"name": "codebook_lookup",
         "description": "Look up a variable definition in the survey codebook",
         "inputSchema": schema(variable="string")},
    ]},
}
print(json.dumps(tools_list_response, indent=2))

{
  "jsonrpc": "2.0",
  "id": 1,
  "result": {
    "tools": [
      {
        "name": "crosstab",
        "description": "Cross-tabulate two survey variables",
        "inputSchema": {
          "type": "object",
          "properties": {
            "row_var": {
              "type": "string"
            },
            "col_var": {
              "type": "string"
            }
          },
          "required": [
            "row_var",
            "col_var"
          ]
        }
      },
      {
        "name": "codebook_lookup",
        "description": "Look up a variable definition in the survey codebook",
        "inputSchema": {
          "type": "object",
          "properties": {
            "variable": {
              "type": "string"
            }
          },
          "required": [
            "variable"
          ]
        }
      }
    ]
  }
}


### A minimal server, protocol-accurate

### Prompt Engineering at the Tool Level: Comments are the new API

> **The Legacy Friction:** In traditional scripting, comments (`#` or `/* */`) are just for human coworkers. If you write a lazy comment, the code still runs fine.
>
> **The RAP Value Proposition:** In MCP, your docstrings (`"""Runs a regression..."""`) are literally scraped by the server, packaged into the JSON Schema Menu, and used as the instruction manual for the AI's brain. If your comment is vague, the LLM will hallucinate and pass the wrong variables. If it is highly specific, the LLM will flawlessly execute it. You are no longer writing comments for humans; you are programming the AI.


In [3]:
class MiniMCPServer:
    """Handles the three core methods. Real servers add auth, streaming and resources."""

    def __init__(self, name: str):
        self.name = name
        self._tools: dict[str, tuple[str, dict, callable]] = {}
        self._resources: dict[str, str] = {}

    def add_tool(self, name, description, input_schema, fn):
        self._tools[name] = (description, input_schema, fn)

    def add_resource(self, uri, content):
        self._resources[uri] = content

    def handle(self, request: dict) -> dict:
        method, rid, params = request["method"], request.get("id"), request.get("params", {})
        try:
            if method == "tools/list":
                result = {"tools": [{"name": n, "description": d, "inputSchema": s}
                                    for n, (d, s, _) in self._tools.items()]}
            elif method == "tools/call":
                name = params["name"]
                if name not in self._tools:
                    raise KeyError(f"unknown tool {name!r}")
                out = self._tools[name][2](**params.get("arguments", {}))
                result = {"content": [{"type": "text", "text": json.dumps(out)}]}
            elif method == "resources/read":
                uri = params["uri"]
                result = {"contents": [{"uri": uri, "text": self._resources[uri]}]}
            else:
                raise ValueError(f"unsupported method {method!r}")
            return {"jsonrpc": "2.0", "id": rid, "result": result}
        except Exception as exc:
            # JSON-RPC errors are data, not exceptions - the client must see them.
            return {"jsonrpc": "2.0", "id": rid,
                    "error": {"code": -32602, "message": f"{type(exc).__name__}: {exc}"}}

In [4]:
server = MiniMCPServer("survey-tools")
server.add_tool("crosstab", "Cross-tabulate two survey variables",
                schema(row_var="string", col_var="string"),
                lambda row_var, col_var: {"chi2": 12.41, "p": 0.006, "n": 2041})
server.add_tool("codebook_lookup", "Look up a variable definition",
                schema(variable="string"),
                lambda variable: {"variable": variable, "label": "Trust in AI (1-5)", "missing": -9})
server.add_resource("survey://codebook/ai_trust", "trusts_ai: 1=none ... 5=complete; -9=missing")

banner("tools/list")
print(json.dumps(server.handle({"jsonrpc": "2.0", "id": 1, "method": "tools/list"}), indent=2)[:400])

banner("tools/call")
show("response", server.handle({"jsonrpc": "2.0", "id": 2, "method": "tools/call",
     "params": {"name": "crosstab", "arguments": {"row_var": "age_group", "col_var": "trusts_ai"}}}))

banner("resources/read")
show("response", server.handle({"jsonrpc": "2.0", "id": 3, "method": "resources/read",
     "params": {"uri": "survey://codebook/ai_trust"}}))

banner("error path - unknown tool")
show("response", server.handle({"jsonrpc": "2.0", "id": 4, "method": "tools/call",
     "params": {"name": "drop_database", "arguments": {}}}))


tools/list
{
  "jsonrpc": "2.0",
  "id": 1,
  "result": {
    "tools": [
      {
        "name": "crosstab",
        "description": "Cross-tabulate two survey variables",
        "inputSchema": {
          "type": "object",
          "properties": {
            "row_var": {
              "type": "string"
            },
            "col_var": {
              "type": "string"
            }
          },
       

tools/call
response:
  {
    "jsonrpc": "2.0",
    "id": 2,
    "result": {
      "content": [
        {
          "type": "text",
          "text": "{\"chi2\": 12.41, \"p\": 0.006, \"n\": 2041}"
        }
      ]
    }
  }

resources/read
response:
  {
    "jsonrpc": "2.0",
    "id": 3,
    "result": {
      "contents": [
        {
          "uri": "survey://codebook/ai_trust",
          "text": "trusts_ai: 1=none ... 5=complete; -9=missing"
        }
      ]
    }
  }

error path - unknown tool
response:
  {
    "jsonrpc": "2.0",
    "id": 4,
    "error": {
      "code": -32602

### Wiring a server into the agent loop

An MCP client turns `tools/list` into the schemas you pass to the model, and routes the
model's requests to `tools/call`. Ten lines — and it is why the loop from notebook 03
needs no changes to gain remote tools.

In [5]:
class MCPClient:
    def __init__(self, server: MiniMCPServer):
        self._server, self._id = server, 0

    def _rpc(self, method, params=None):
        self._id += 1
        resp = self._server.handle({"jsonrpc": "2.0", "id": self._id,
                                    "method": method, "params": params or {}})
        if "error" in resp:
            raise RuntimeError(resp["error"]["message"])
        return resp["result"]

    def as_toolbox(self) -> ToolBox:
        tb = ToolBox()
        for t in self._rpc("tools/list")["tools"]:
            name = t["name"]
            tb.register(Tool(name, t["description"], t["inputSchema"],
                             lambda _n=name, **kw: self._rpc("tools/call",
                                                             {"name": _n, "arguments": kw})))
        return tb

remote_tools = MCPClient(server).as_toolbox()
print(remote_tools.describe())
show("Called over the protocol", remote_tools.call("codebook_lookup", {"variable": "trusts_ai"}))

- crosstab(row_var, col_var): Cross-tabulate two survey variables
- codebook_lookup(variable): Look up a variable definition
Called over the protocol:
  {
    "content": [
      {
        "type": "text",
        "text": "{\"variable\": \"trusts_ai\", \"label\": \"Trust in AI (1-5)\", \"missing\": -9}"
      }
    ]
  }


### The real SDK

The `mcp` package is already in requirements.txt. A production server is:

```python
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("survey-tools")

@mcp.tool()
def crosstab(row_var: str, col_var: str) -> dict:
    """Cross-tabulate two survey variables."""
    ...

mcp.run(transport="stdio")
```

Point Claude Desktop at that command and the tool appears in the client. The JSON-RPC
you just hand-wrote is what flows over stdio.

In [6]:
try:
    import mcp
    print(f"mcp {getattr(mcp, '__version__', 'installed')} is available.")
    print("Exercise: port MiniMCPServer to FastMCP and connect it to Claude Desktop.")
except ImportError:
    print("mcp not installed - pip install mcp")

mcp installed is available.
Exercise: port MiniMCPServer to FastMCP and connect it to Claude Desktop.


---
### Try it yourself

1. Add `prompts/list` — MCP servers can ship reusable prompt templates too.
2. `tools/call` currently trusts its arguments. Add the pydantic gate from notebook 02.
3. Why does MCP separate *resources* (read a document) from *tools* (run an action)?